# Greek City-States — active individuals

Three views of the individuals matched to the **Greek City-States** Cliopatria polity, using `individuals_floruit_period`.

Counting rule (uniform): an individual is counted in every bin their floruit interval overlaps. The `floruit_year` column is not used.

1. **Per century** — every individual; counted in each century their floruit period overlaps.
2. **Per year** — only year-range floruits (e.g. `-589..-564`); counted in each year their range covers.
3. **Comparison at the century scale** — two series side by side on the same century anchors:
   - **All individuals (overlap)** — same logic as panel 1.
   - **Year-range only, rounded to nearest century** — each year-range individual is placed at one century, with the midpoint rounded to the nearest 100 (so a midpoint at 1850 goes to 1900, 1849 goes to 1800; -491.5 goes to -500). This shows how much extra signal the per-century overlap counting captures, on top of the year-range subset rounded to centuries.

In [ ]:
# === notebook config (auto-managed; edit values, not the tag) ===
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Database
DB_PATH = "../data/humans_clean.duckdb"

# Figure style — minimal, Nature/Science publication standard
FIGSIZE = (8, 5)
DPI = 120
FONT_TITLE = 16
FONT_LABEL = 13
FONT_TICK = 11
FONT_LEGEND = 10

# Light, restrained palette (avoid AI-slop saturation)
COLOR_PRIMARY = "#2171b5"
COLOR_SECONDARY = "#b5542a"
COLOR_NEUTRAL = "#7f7f7f"
COLOR_LIGHT = "#d9d9d9"
COLOR_ACCENT = "#6a9e3a"
PALETTE = [COLOR_PRIMARY, COLOR_SECONDARY, COLOR_ACCENT, COLOR_NEUTRAL, COLOR_LIGHT]

import matplotlib as _mpl
_mpl.rcParams.update({
    "figure.figsize": FIGSIZE,
    "figure.dpi": DPI,
    "axes.titlesize": FONT_TITLE,
    "axes.labelsize": FONT_LABEL,
    "xtick.labelsize": FONT_TICK,
    "ytick.labelsize": FONT_TICK,
    "legend.fontsize": FONT_LEGEND,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})


## 1. Imports and configuration

In [ ]:
import re
import duckdb
import polars as pl
import matplotlib.pyplot as plt

DB = DB_PATH
POLITY = 'Greek City-States'
# Display range: 800 BCE to 83 BCE
YEAR_MIN, YEAR_MAX = -800, -83

CENTURY_COLOR = '#2f5b8a'   # dark blue
DECADE_COLOR  = '#4c78a8'   # medium blue
YEAR_COLOR    = '#7ba4c8'   # light blue

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'font.size': 12,
    'font.family': 'DejaVu Sans',
})

np.random.seed(0)


## 2. Load Greek City-States individuals

`polity_name` may be a `;`-separated list, so we test with `LIKE '%;Greek City-States;%'`.

In [ ]:
conn = duckdb.connect(DB, read_only=True)
df = conn.execute("""
    SELECT DISTINCT ic.wikidata_id,
           fp.method,
           fp.floruit_period,
           fp.floruit_period_start
    FROM individuals_cliopatria ic
    JOIN individuals_floruit_period fp USING (wikidata_id)
    WHERE ';' || ic.polity_name || ';' LIKE ?
      AND fp.method IS NOT NULL
      AND fp.floruit_period IS NOT NULL
    """, [f'%;{POLITY};%']).pl()
conn.close()
print(f'Total individuals matched to {POLITY}: {df.height:,}')
df.head()

## 3. Parse `floruit_period` into a `[start, end]` year interval

Three formats appear: explicit year ranges (`-552--527`), single century (`5th c. BC`), and rare multi-century ranges (`7th-6th c. BC`). Convention: `n`th c. AD = `(n-1)*100+1 .. n*100`; `n`th c. BC = `-(n*100) .. -((n-1)*100+1)`.

In [ ]:
_RANGE_RE = re.compile(r'^(-?\d+)-(-?\d+)$')
_CENT_RE  = re.compile(r'^(\d+)\D*\s*c\.\s*(AD|BC)$', re.IGNORECASE)
_CENT2_RE = re.compile(r'^(\d+)\D*-\s*(\d+)\D*\s*c\.\s*(AD|BC)$', re.IGNORECASE)

def _cent_bounds(n, era):
    if era == 'AD':
        return (n - 1) * 100 + 1, n * 100
    return -(n * 100), -((n - 1) * 100 + 1)

def parse_period(period):
    if period is None:
        return {'flo_start': None, 'flo_end': None, 'period_kind': None}
    s = period.strip()
    m = _RANGE_RE.match(s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return {'flo_start': min(a, b), 'flo_end': max(a, b), 'period_kind': 'range'}
    m = _CENT2_RE.match(s)
    if m:
        n1, n2, era = int(m.group(1)), int(m.group(2)), m.group(3).upper()
        a1, b1 = _cent_bounds(n1, era)
        a2, b2 = _cent_bounds(n2, era)
        return {'flo_start': min(a1, a2), 'flo_end': max(b1, b2), 'period_kind': 'century'}
    m = _CENT_RE.match(s)
    if m:
        a, b = _cent_bounds(int(m.group(1)), m.group(2).upper())
        return {'flo_start': a, 'flo_end': b, 'period_kind': 'century'}
    return {'flo_start': None, 'flo_end': None, 'period_kind': None}

_parse_schema = pl.Struct([
    pl.Field('flo_start', pl.Int64),
    pl.Field('flo_end', pl.Int64),
    pl.Field('period_kind', pl.Utf8),
])

df = (
    df.with_columns(
        pl.col('floruit_period').map_elements(parse_period, return_dtype=_parse_schema).alias('_parsed')
    )
    .unnest('_parsed')
    .with_columns(
        pl.when(pl.col('method').is_in(['birth_century', 'death_century']))
          .then(pl.lit('century'))
          .otherwise(pl.lit('decade'))
          .alias('precision'),
    )
    .drop_nulls(subset=['flo_start', 'flo_end'])
    .with_columns(
        pl.col('flo_start').cast(pl.Int64),
        pl.col('flo_end').cast(pl.Int64),
    )
    .with_columns(
        # Representative year for the century / 50-year panels — keeps using
        # floruit_period_start when available so that century-string rows
        # retain their stamped anchor.
        pl.coalesce(
            pl.col('floruit_period_start').cast(pl.Float64),
            ((pl.col('flo_start') + pl.col('flo_end')) / 2).cast(pl.Float64),
        ).alias('rep_year'),
        # Per-year panel uses ONLY year-range floruit_period strings.
        pl.when(pl.col('period_kind') == 'range')
          .then(((pl.col('flo_start') + pl.col('flo_end')) / 2).round(0))
          .otherwise(None)
          .alias('year_anchor'),
    )
)

print(f'Decade-precise: {df.filter(pl.col("precision") == "decade").height:,}  |  '
      f'Century-only: {df.filter(pl.col("precision") == "century").height:,}')
print(f'Period kind  : {dict(df.group_by("period_kind").agg(pl.len().alias("n")).iter_rows())}')
df.select(['wikidata_id', 'method', 'floruit_period', 'period_kind',
           'rep_year', 'year_anchor', 'precision']).head()

### Figure: Greek City-States — per century, per year, and per-century comparison

Top: per century (all individuals via overlap).
Middle: per year (year-range floruits via overlap).
Bottom: side-by-side comparison at the century scale — overlap-counted vs year-range rounded to the nearest century.

In [ ]:
import math
from matplotlib.ticker import PercentFormatter

# Counting rule: each individual contributes to EVERY bin their floruit
# interval [flo_start, flo_end] overlaps.

def overlap_counts(rows, bin_starts, width):
    """Number of unique individuals active in each bin [B, B+width-1].
    `rows` is a polars DataFrame with columns flo_start, flo_end, wikidata_id."""
    a = rows['flo_start'].to_numpy()
    b = rows['flo_end'].to_numpy()
    wid = rows['wikidata_id'].to_numpy()
    out_bins = []
    out_n = []
    for B in bin_starts:
        m = (a <= B + width - 1) & (b >= B)
        out_bins.append(B)
        out_n.append(int(np.unique(wid[m]).size))
    return pl.DataFrame({'bin': out_bins, 'n': out_n})

def round_to_century(x):
    """Round to nearest 100 with half rounding toward +infinity."""
    return int(math.floor(x / 100 + 0.5) * 100)

def normalize(arr):
    total = arr.sum()
    return arr / total if total else arr

# --- per-century counts (all individuals, overlap) ---
century_starts = list(range(int(np.floor(YEAR_MIN / 100) * 100),
                            int(np.floor(YEAR_MAX / 100) * 100) + 1, 100))
century_counts = overlap_counts(df, century_starts, 100)['n'].to_numpy()
N_century = int(df['wikidata_id'].n_unique())

# --- per-year counts (year-range floruits only, overlap) ---
range_src = df.filter(pl.col('period_kind') == 'range')
year_starts = list(range(YEAR_MIN, YEAR_MAX + 1))
year_counts = overlap_counts(range_src, year_starts, 1)['n'].to_numpy()
N_year = int(range_src['wikidata_id'].n_unique())

# --- per-century from year-range, rounded to nearest century ---
range_src = range_src.with_columns(
    ((pl.col('flo_start') + pl.col('flo_end')) / 2).alias('midpoint'),
).with_columns(
    pl.col('midpoint').map_elements(round_to_century, return_dtype=pl.Int64).alias('cent_rounded')
)
rounded_in_range = range_src.filter(pl.col('cent_rounded').is_in(century_starts))
rounded_century_counts_df = (
    pl.DataFrame({'cent_rounded': century_starts})
    .join(
        rounded_in_range.group_by('cent_rounded').agg(pl.col('wikidata_id').n_unique().alias('n')),
        on='cent_rounded', how='left',
    )
    .with_columns(pl.col('n').fill_null(0))
    .sort('cent_rounded')
)
rounded_century_counts = rounded_century_counts_df['n'].to_numpy()
N_rounded = int(rounded_in_range['wikidata_id'].n_unique())

# Bottom-panel normalisation (each series sums to 100%)
century_share         = normalize(century_counts.astype(float))
rounded_century_share = normalize(rounded_century_counts.astype(float))

print(f'Per century (overlap, all)        : N = {N_century:,}')
print(f'Per year (year-range, overlap)    : N = {N_year:,}')
print(f'Per century (year-range, rounded) : N = {N_rounded:,}')

# Main periods of ancient Greek history (BCE values are negative)
PERIODS = [
    ('Archaic',     -800, -480, '#e8e2d4'),
    ('Classical',   -480, -323, '#d8e3ec'),
    ('Hellenistic', -323,  -31, '#dfead8'),
    ('Roman',        -31,   83, '#ecdcdc'),
]

def draw_periods(ax, ymax):
    for name, a, b, color in PERIODS:
        a_clip, b_clip = max(a, YEAR_MIN), min(b, YEAR_MAX)
        if b_clip <= a_clip:
            continue
        ax.axvspan(a_clip, b_clip, color=color, alpha=0.55, zorder=0)
        ax.text((a_clip + b_clip) / 2, ymax * 1.02, name,
                ha='center', va='bottom', fontsize=12, color='#444')
    for _, a, _, _ in PERIODS[1:]:
        if YEAR_MIN < a < YEAR_MAX:
            ax.axvline(a, color='#999', linewidth=0.7,
                       linestyle=':', zorder=0.5)

# --- combined figure ---
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 11.4), sharex=True)

# Panel 1 — per century (absolute counts)
top_y1 = float(century_counts.max()) * 1.05
ax1.bar(np.array(century_starts) + 50, century_counts, width=90,
        color=CENTURY_COLOR, alpha=0.9, edgecolor='white', linewidth=0.6,
        zorder=2)
draw_periods(ax1, top_y1)
ax1.set_ylim(top=top_y1 * 1.18)
ax1.set_ylabel('Active individuals\n(per century)', fontsize=14)
ax1.tick_params(axis='both', labelsize=13, labelbottom=True)
ax1.text(0.02, 0.95, f'N = {N_century:,}', transform=ax1.transAxes,
         ha='left', va='top', fontsize=14, color='black',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                   edgecolor='lightgray'))

# Panel 2 — per year (absolute counts)
top_y2 = float(year_counts.max()) * 1.05
ax2.bar(np.array(year_starts), year_counts, width=1.0,
        color=YEAR_COLOR, alpha=0.95, edgecolor='none', zorder=2)
draw_periods(ax2, top_y2)
ax2.set_ylim(top=top_y2 * 1.18)
ax2.set_ylabel('Active individuals\n(per year)', fontsize=14)
ax2.tick_params(axis='both', labelsize=13, labelbottom=True)
ax2.text(0.02, 0.95, f'N = {N_year:,}', transform=ax2.transAxes,
         ha='left', va='top', fontsize=14, color='black',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                   edgecolor='lightgray'))

# Panel 3 — comparison at the century scale, NORMALISED (each series sums to 100%)
top_y3 = float(max(century_share.max(), rounded_century_share.max())) * 1.05
bar_w = 38
xs = np.array(century_starts) + 50
ax3.bar(xs - bar_w / 2, century_share, width=bar_w,
        color=CENTURY_COLOR, alpha=0.9, edgecolor='white', linewidth=0.6,
        zorder=2, label='Century')
ax3.bar(xs + bar_w / 2, rounded_century_share, width=bar_w,
        color=YEAR_COLOR, alpha=0.95, edgecolor='white', linewidth=0.6,
        zorder=2, label='Year rounded')
draw_periods(ax3, top_y3)
ax3.set_ylim(top=top_y3 * 1.18)
ax3.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax3.set_ylabel('Share of active\nindividuals (per century)', fontsize=14)
ax3.set_xlabel('Year', fontsize=14)
ax3.tick_params(axis='both', labelsize=13, labelbottom=True)
ax3.legend(frameon=False, fontsize=12, loc='upper right')

ax3.set_xlim(YEAR_MIN, YEAR_MAX)
fig.tight_layout()
plt.show()